In [1]:
from generate_utils import load_SELaC, nucleus_token_by_token_generate
import torch
from torch.utils.data import DataLoader, ConcatDataset
import numpy as np
import pickle
from tqdm import tqdm
from GridMLM_tokenizers import CSGridMLMTokenizer
from data_utils import CSGridMLMDataset, CSGridMLM_collate_fn
import matplotlib.pyplot as plt
from copy import deepcopy
import os
from dotenv import load_dotenv

/home/maximos/miniconda3/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
batchsize = 8

In [3]:
train_gjt = os.getenv('TRAIN_GJT')
train_nott = os.getenv('TRAIN_NOTT')

val_gjt = os.getenv('VAL_GJT')
val_nott = os.getenv('VAL_NOTT')

In [4]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [5]:
train_dataset_gjt = CSGridMLMDataset(train_gjt, tokenizer, frontloading=True, name_suffix='Q4_L80_bar_PC')
train_dataset_nott = CSGridMLMDataset(train_nott, tokenizer, frontloading=True, name_suffix='Q4_L80_bar_PC')

val_dataset_gjt = CSGridMLMDataset(val_gjt, tokenizer, frontloading=True, name_suffix='Q4_L80_bar_PC')
val_dataset_nott = CSGridMLMDataset(val_nott, tokenizer, frontloading=True, name_suffix='Q4_L80_bar_PC')


Loading data file.
Loading data file.
Loading data file.
Loading data file.


In [6]:
gjtTrainLoader = DataLoader(
    train_dataset_gjt,
    batch_size=batchsize,
    shuffle=False,
    drop_last=False,
    collate_fn=CSGridMLM_collate_fn
)

nottTrainLoader = DataLoader(
    train_dataset_nott,
    batch_size=batchsize,
    shuffle=False,
    drop_last=False,
    collate_fn=CSGridMLM_collate_fn
)

gjtValLoader = DataLoader(
    val_dataset_gjt,
    batch_size=batchsize,
    shuffle=False,
    drop_last=False,
    collate_fn=CSGridMLM_collate_fn
)

nottValLoader = DataLoader(
    val_dataset_nott,
    batch_size=batchsize,
    shuffle=False,
    drop_last=False,
    collate_fn=CSGridMLM_collate_fn
)

In [7]:
device_name = 'cuda:1'

if device_name == 'cpu':
    device = torch.device('cpu')
else:
    if torch.cuda.is_available():
        device = torch.device(device_name)
    else:
        print('Selected device not available: ' + device_name)
# end device selection

In [8]:
model = load_SELaC(
    tokenizer,
    device,
    guidance_dim=512,
    d_model=512,
    checkpoint_path=None,
)
model.eval()

SELaCModel(
  (melody_proj): Linear(in_features=13, out_features=512, bias=True)
  (harmony_embedding): Embedding(355, 512)
  (dropout): Dropout(p=0.3, inplace=False)
  (encoder_layers): ModuleList(
    (0-7): 8 x TransformerEncoderLayerWithAttn(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
      )
      (linear1): Linear(in_features=512, out_features=2048, bias=True)
      (dropout): Dropout(p=0.3, inplace=False)
      (linear2): Linear(in_features=2048, out_features=512, bias=True)
      (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (dropout1): Dropout(p=0.3, inplace=False)
      (dropout2): Dropout(p=0.3, inplace=False)
    )
  )
  (output_head): Linear(in_features=512, out_features=355, bias=True)
  (input_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (output_norm): LayerNorm((512,), 

In [9]:
gjt_train_latent_steps_list = []
gjt_val_latent_steps_list = []

nott_train_latent_steps_list = []
nott_val_latent_steps_list = []

In [10]:
with torch.inference_mode():
    for batch in gjtTrainLoader:
        _, latent_steps = model(
            batch['pianoroll'].to(device),
            batch['harmony_ids'].to(device),
            get_layers_output=True
        )
        gjt_train_latent_steps_list.append(
            {layer: activations.cpu() for layer, activations in latent_steps.items()}
        )
    for batch in gjtValLoader:
        _, latent_steps = model(
            batch['pianoroll'].to(device),
            batch['harmony_ids'].to(device),
            get_layers_output=True
        )
        gjt_val_latent_steps_list.append(
            {layer: activations.cpu() for layer, activations in latent_steps.items()}
        )
    for batch in nottTrainLoader:
        _, latent_steps = model(
            batch['pianoroll'].to(device),
            batch['harmony_ids'].to(device),
            get_layers_output=True
        )
        nott_train_latent_steps_list.append(
            {layer: activations.cpu() for layer, activations in latent_steps.items()}
        )
    for batch in nottValLoader:
        _, latent_steps = model(
            batch['pianoroll'].to(device),
            batch['harmony_ids'].to(device),
            get_layers_output=True
        )
        nott_val_latent_steps_list.append(
            {layer: activations.cpu() for layer, activations in latent_steps.items()}
        )

In [11]:
print(len(gjt_train_latent_steps_list))
print(gjt_train_latent_steps_list[0][0].shape)

65
torch.Size([8, 1, 80, 512])


In [12]:
gjt_train_latent_steps_dict = {}
gjt_val_latent_steps_dict = {}

nott_train_latent_steps_dict = {}
nott_val_latent_steps_dict = {}

for k in range(8):
    gjt_train_latent_steps_dict[k] = np.concatenate(
        [
            torch.flatten(
                batch_steps[k].squeeze(1),
                start_dim=0,
                end_dim=1
            ).numpy()
            for batch_steps in gjt_train_latent_steps_list
        ],
        axis=0
    )
    gjt_val_latent_steps_dict[k] = np.concatenate(
        [
            torch.flatten(
                batch_steps[k].squeeze(1),
                start_dim=0,
                end_dim=1
            ).numpy()
            for batch_steps in gjt_val_latent_steps_list
        ],
        axis=0
    )
    nott_train_latent_steps_dict[k] = np.concatenate(
        [
            torch.flatten(
                batch_steps[k].squeeze(1),
                start_dim=0,
                end_dim=1
            ).numpy()
            for batch_steps in nott_train_latent_steps_list
        ],
        axis=0
    )
    nott_val_latent_steps_dict[k] = np.concatenate(
        [
            torch.flatten(
                batch_steps[k].squeeze(1),
                start_dim=0,
                end_dim=1
            ).numpy()
            for batch_steps in nott_val_latent_steps_list
        ],
        axis=0
    )

In [13]:
print(gjt_train_latent_steps_dict[0].shape)
print(gjt_val_latent_steps_dict[0].shape)
print(nott_train_latent_steps_dict[0].shape)
print(nott_val_latent_steps_dict[0].shape)

(41520, 512)
(2240, 512)
(67440, 512)
(3600, 512)


In [14]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

In [15]:
os.makedirs('figs', exist_ok=True)
os.makedirs('figs/pca', exist_ok=True)

for k in range(8):
    tmp_pca = PCA(n_components=2)

    gjt_train = gjt_train_latent_steps_dict[k]
    gjt_val = gjt_val_latent_steps_dict[k]
    nott_train = nott_train_latent_steps_dict[k]
    nott_val = nott_val_latent_steps_dict[k]

    group_sizes = [len(gjt_train), len(gjt_val), len(nott_train), len(nott_val)]
    all_latent_steps = np.concatenate(
        [gjt_train, gjt_val, nott_train, nott_val],
        axis=0
    )
    all_latent_steps_2d = tmp_pca.fit_transform(all_latent_steps)
    gjt_train_2d, gjt_val_2d, nott_train_2d, nott_val_2d = np.split(
        all_latent_steps_2d,
        np.cumsum(group_sizes)[:-1],
        axis=0
    )

    plt.figure(figsize=(8, 6))
    plt.scatter(gjt_train_2d[:, 0], gjt_train_2d[:, 1], c='firebrick', marker='|', label='GJT train', alpha=0.01)
    plt.scatter(gjt_val_2d[:, 0], gjt_val_2d[:, 1], c='fuchsia', marker='1', label='GJT validation', alpha=0.1)
    plt.scatter(nott_train_2d[:, 0], nott_train_2d[:, 1], c='blue', marker='_', label='NoTT train', alpha=0.01)
    plt.scatter(nott_val_2d[:, 0], nott_val_2d[:, 1], c='cyan', marker='3', label='NoTT validation', alpha=0.1)
    plt.title(f'PCA of latent layer {k}')
    plt.xlabel('Principal component 1')
    plt.ylabel('Principal component 2')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'figs/pca/pca_{k}.png', dpi=300)
    plt.close()

In [16]:
import numba
numba.set_num_threads(4)

os.makedirs('figs', exist_ok=True)
os.makedirs('figs/umap', exist_ok=True)

for k in range(8):
    tmp_umap = umap.UMAP(
        n_components=2,
        n_jobs=4,
        random_state=None,
        n_neighbors=100,
        min_dist=0.5,
        init="random"
    )

    gjt_train = gjt_train_latent_steps_dict[k]
    gjt_val = gjt_val_latent_steps_dict[k]
    nott_train = nott_train_latent_steps_dict[k]
    nott_val = nott_val_latent_steps_dict[k]

    group_sizes = [len(gjt_train), len(gjt_val), len(nott_train), len(nott_val)]
    all_latent_steps = np.concatenate(
        [gjt_train, gjt_val, nott_train, nott_val],
        axis=0
    )
    all_latent_steps_2d = tmp_umap.fit_transform(all_latent_steps)
    gjt_train_2d, gjt_val_2d, nott_train_2d, nott_val_2d = np.split(
        all_latent_steps_2d,
        np.cumsum(group_sizes)[:-1],
        axis=0
    )

    plt.figure(figsize=(8, 6))
    plt.scatter(gjt_train_2d[:, 0], gjt_train_2d[:, 1], c='firebrick', marker='|', label='GJT train', alpha=0.01)
    plt.scatter(gjt_val_2d[:, 0], gjt_val_2d[:, 1], c='fuchsia', marker='1', label='GJT validation', alpha=0.1)
    plt.scatter(nott_train_2d[:, 0], nott_train_2d[:, 1], c='blue', marker='_', label='NoTT train', alpha=0.01)
    plt.scatter(nott_val_2d[:, 0], nott_val_2d[:, 1], c='cyan', marker='3', label='NoTT validation', alpha=0.1)
    plt.title(f'UMAP of latent layer {k}')
    plt.xlabel('Component 1')
    plt.ylabel('Component 2')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'figs/umap/umap_{k}.png', dpi=300)
    plt.close()